# 03 — Fact Tables

**Project:** World Forest Change & GDP Correlation Analysis
**Database:** db_forestgdp (MSSQL Server)
**Team:** [Kucerova Kristina], [Kmetova Barbara]
**Date:** May 2026

## Purpose
This notebook builds the fact tables in the `dbo` schema.
Dimension tables from 02_dim_tables must be built before running this notebook.

## Tables to build
**dbo.fact_forest** — sourced from raw.Forest_year — forest coverage % per country per year

**dbo.fact_gdp** — sourced from raw.GDP — GDP per capita per country per year

## Notes
- Analysis range filtered to 1990—2024 where both tables overlap
- Foreign keys linked to dbo.dim_country and dbo.dim_year

In [9]:
-- Create fact_forest
CREATE TABLE dbo.fact_forest (
    id INT IDENTITY(1,1) PRIMARY KEY,
    country_code VARCHAR(10) NOT NULL,
    year INT NOT NULL,
    forest_pct DECIMAL(10,2)
);

Commands completed successfully.

Total execution time: 00:00:00.022

In [ ]:
-- Populate fact_forest with forest coverage data
-- Filtered to analysis range 1990-2025
INSERT INTO dbo.fact_forest (country_code, year, forest_pct)
SELECT
    f.Code AS country_code,
    f.Year AS year,
    f.Forest_percentage AS forest_pct
FROM raw.Forest_year f
WHERE f.Year BETWEEN 1990 AND 2025;

(7860 rows affected)

Total execution time: 00:00:00.071

In [16]:
-- Create fact_gdp
CREATE TABLE dbo.fact_gdp (
    id INT IDENTITY(1,1) PRIMARY KEY,
    country_code VARCHAR(10) NOT NULL,
    year INT NOT NULL,
    gdp_per_capita DECIMAL(15,2));

Commands completed successfully.

Total execution time: 00:00:00.019

In [17]:
-- Populate fact_gdp with GDP per capita data
-- Filtered to analysis range 1990-2025
INSERT INTO dbo.fact_gdp (country_code, year, gdp_per_capita)
SELECT
    g.[Country Code]    AS country_code,  -- ISO3 code, reference to dim_country
    g.Year              AS year,          -- Reference to dim_year
    g.GDP               AS gdp_per_capita_usd -- GDP per capita in current US$
FROM raw.GDP g
WHERE g.Year BETWEEN 1990 AND 2024;

(9310 rows affected)

Total execution time: 00:00:00.084

In [22]:
-- Verify fact_forest: total rows and year range
SELECT COUNT(*) AS total_rows,
       MIN(year) AS year_from,
       MAX(year) AS year_to
FROM dbo.fact_forest;

-- Verify fact_gdp: total rows and year range
SELECT COUNT(*) AS total_rows,
       MIN(year) AS year_from,
       MAX(year) AS year_to
FROM dbo.fact_gdp;

(1 row affected)
(1 row affected)

total_rows | year_from | year_to
-----------+-----------+--------
7860       | 1990      | 2025   
(1 row)

total_rows | year_from | year_to
-----------+-----------+--------
9310       | 1990      | 2024   
(1 row)

Total execution time: 00:00:00.036

In [ ]:
-- Test join across all tables that checks that dim_country and dim_year link correctly to both fact tables. Teste for randon year  2020.
SELECT
    c.country_name,
    c.region,
    c.income_group,
    y.year,
    f.forest_pct,
    g.gdp_per_capita
FROM dbo.dim_country c
JOIN dbo.fact_forest f  ON f.country_code = c.country_code
JOIN dbo.fact_gdp g     ON g.country_code = c.country_code
                       AND g.year = f.year
JOIN dbo.dim_year y     ON y.year = f.year
WHERE y.year = 2020
ORDER BY c.country_name;

(210 rows affected)

country_name                     | region                     | income_group        | year | forest_pct | gdp_per_capita
---------------------------------+----------------------------+---------------------+------+------------+---------------
Afghanistan                      | South Asia                 | Low income          | 2020 | 1.85       | 510.79        
Albania                          | Europe & Central Asia      | Upper middle income | 2020 | 34.34      | 6027.91       
Algeria                          | Middle East & North Africa | Lower middle income | 2020 | 0.71       | 3743.54       
American Samoa                   | East Asia & Pacific        | High income         | 2020 | 79.25      | 14489.26      
Andorra                          | Europe & Central Asia      | High income         | 2020 | 38.64      | 37361.11      
Angola                           | Sub-Saharan Africa         | Lower middle income | 2020 | 52.79      | 1759.36       
Antigua and